##Import Libraries

In [0]:
from pyspark.sql.functions import (
    col, when, lit,
    abs, sum, min, max,
    length, trim, lower, upper, regexp_replace,
    to_date, to_timestamp, current_date, current_timestamp, date_format
)

from datetime import datetime, date
import uuid

print('functions imported successfully')

##Read From Bronze Layer

In [0]:
df=spark.read.table('workspace.bronze.crypto_raw')
print("read succesfully!")

##Checking Schema

In [0]:
print('schema check -->')
df.printSchema()
print('*'*40)
print('data sample check...')
display(df.limit(5))



##Drop unnecessary columns,Cast data types,Rename columns

In [0]:
#not needed/meaningfull for Analysis
df=df.drop("image","market_cap_change_24h",'price_change_24h',"fully_diluted_valuation")
print('dropped columns')

#handling dtypes
from pyspark.sql.functions import col

df = df \
    .withColumn("id", col("id").cast("string")) \
    .withColumn("symbol", col("symbol").cast("string")) \
    .withColumn("name", col("name").cast("string")) \
    .withColumn("current_price", col("current_price").cast("double")) \
    .withColumn("market_cap", col("market_cap").cast("double")) \
    .withColumn("market_cap_rank", col("market_cap_rank").cast("integer")) \
    .withColumn("total_volume", col("total_volume").cast("double")) \
    .withColumn("high_24h", col("high_24h").cast("double")) \
    .withColumn("low_24h", col("low_24h").cast("double")) \
    .withColumn("price_change_percentage_24h", col("price_change_percentage_24h").cast("double")) \
    .withColumn("market_cap_change_percentage_24h", col("market_cap_change_percentage_24h").cast("double")) \
    .withColumn("circulating_supply", col("circulating_supply").cast("double")) \
    .withColumn("total_supply", col("total_supply").cast("double")) \
    .withColumn("max_supply", col("max_supply").cast("double")) \
    .withColumn("ath", col("ath").cast("double")) \
    .withColumn("ath_change_percentage", col("ath_change_percentage").cast("double")) \
    .withColumn("ath_date", col("ath_date").cast("timestamp")) \
    .withColumn("atl", col("atl").cast("double")) \
    .withColumn("atl_change_percentage", col("atl_change_percentage").cast("double")) \
    .withColumn("atl_date", col("atl_date").cast("timestamp")) \
    .withColumn("last_updated", col("last_updated").cast("timestamp"))

print('data types handled successfully')
print('*'*40)  

#renaming columns 
df = df \
    .withColumnRenamed("high_24h", "high_price_24h") \
    .withColumnRenamed("low_24h", "low_price_24h") \
    .withColumnRenamed("price_change_percentage_24h", "price_change_pct_24h") \
    .withColumnRenamed("market_cap_change_percentage_24h", "market_cap_change_pct_24h") \
    .withColumnRenamed("ath", "all_time_high") \
    .withColumnRenamed("ath_change_percentage", "ath_change_pct") \
    .withColumnRenamed("ath_date", "all_time_high_date") \
    .withColumnRenamed("atl", "all_time_low") \
    .withColumnRenamed("atl_change_percentage", "atl_change_pct") \
    .withColumnRenamed("atl_date", "all_time_low_date") 

print('renamed columns successfully')
print('*'*40)  

print('check names,dtypes changes')
df.printSchema()


##nulls handling

In [0]:
#null profiling
null_count = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
null_count.display()

#check null pct
total_rows=df.count()
null_pct=null_count.collect()[0].asDict()
for col_name,count in null_pct.items():
    percent=(count/total_rows)*100
    #print(f"{col_name} has {percent:.2f}% nulls")

spark.createDataFrame(null_pct.items(),['column','null_count_pct']).show(truncate=False)
print('*'*40)

#rules
FILL_WITH_ZERO = ["market_cap_change_pct_24h","total_volume","price_change_pct_24h"]
DROP_IF_NULL_COL = ["id", "name","symbol","last_updated"]

#apply rules
print('before drop :',df.count())
df=df.na.drop(subset=DROP_IF_NULL_COL)
print('after drop :',df.count())
print('*'*40)
existing_cols = [c for c in FILL_WITH_ZERO if c in df.columns]
df = df.na.fill({c: 0 for c in existing_cols})
df = df.drop("roi") #as its 89% null

#validation check
remaining_nulls = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)for c in df.columns])
remaining_nulls.display()
print("Null handling completed")

##duplicates handling

In [0]:
print('before duplicate drop :',df.count())
df=df.dropDuplicates(['id'])
print('dropped successfully')
print('after duplicate drop :',df.count())
print('*'*40)


##standardize strings

In [0]:
string_rules = {
    'id': lambda c: trim(c),
    'name': lambda c: lower(trim(c)),
    'symbol': lambda c: upper(trim(c)),
}

for colname,rule in string_rules.items():
    df = df.withColumn(colname, rule(col(colname)))

print('String standardization applied successfully!')


##logical validations

In [0]:
print('-'*40)
print('Data validation checks')
print('-'*40)
validation_results=[]
total = df.count()
def validate(colname,rule,condition) :
    invalid= df.filter(~condition).count()
    validation_results.append({
        'column' : colname,
        'rule' : rule,
        'invalid_count' : invalid,
        'invalid_pct' : invalid / total if total != 0 else None
    })

validate("all_time_high","all_time_high >= current_price",col("all_time_high") >= col("current_price"))
validate("all_time_low", "all_time_low <= current_price",col("all_time_low") <= col("current_price"))
validate("all_time_high","all_time_high > all_time_low",col("all_time_high") > col("all_time_low"))
validate("all_time_high","all_time_high > 0",col("all_time_high") > 0)
validate("all_time_low","all_time_low > 0",col("all_time_low") > 0)
validate("current_price","current_price > 0",col("current_price") >0)
validate('all_time_high_date','all_time_high_date <= Current_timestamp',col('all_time_high_date').isNotNull() &
(col('all_time_high_date') <= current_timestamp()))

validate('all_time_low_date','all_time_low_date < current_timestamp',col('all_time_low_date').isNotNull() &
(col('all_time_low_date') <= current_timestamp()))

validate('circulating_supply','circulating_supply > 0',col('circulating_supply') > 0)
validate('max_supply','max_supply > 0',col('max_supply') > 0)
validate('total_supply','total_supply > 0',col('total_supply') > 0)

validate("supply_hierarchy","circulating <= total <= max (when max exists)",
    col("circulating_supply").isNotNull() &
    col("total_supply").isNotNull() &
    ((col("circulating_supply") <= col("total_supply")) &
    (col("max_supply").isNull() | (col("total_supply") <= col("max_supply")))))

validate("market_cap","market_cap > 0",col("market_cap") > 0)
validate("market_cap_consistency","market_cap ≈ price * circulating_supply",
    col("market_cap").isNotNull() &
    (col("market_cap") > 0) &
    col("current_price").isNotNull() &
    col("circulating_supply").isNotNull() &
    (
        (abs(col("market_cap") - (col("current_price") * col("circulating_supply"))) 
        / col("market_cap")) < 0.05
    )
)

validate("market_cap_rank","market_cap_rank >= 1",col("market_cap_rank") >= 1)
rank_dup = df.groupBy("market_cap_rank").count().filter(col("count") > 1).count()
validation_results.append({
    "column": "market_cap_rank",
    "rule": "rank must be unique",
    "invalid_count": rank_dup,
    "invalid_pct": rank_dup / total if total != 0 else None
})

validate("total_volume","total_volume => 0",col("total_volume") >= 0)
validate("market_cap_change_pct_24h","between -100 and 100",
col("market_cap_change_pct_24h").isNull() | col("market_cap_change_pct_24h").between(-100, 100))

validate("id","id not null",col("id").isNotNull())
id_dup = df.groupBy("id").count().filter(col("count") > 1).count()
validation_results.append({
    "column": "id",
    "rule": "id must be unique",
    "invalid_count": id_dup,
    "invalid_pct": id_dup / total if total != 0 else None
})

validate("name","name  not null",col("name").isNotNull())
validate("symbol","symbol  not null",col("symbol").isNotNull())
validate('symbol',"symbol must be short (1-10)charcters", (length(col('symbol')) <= 10) &(length(col('symbol')) >= 1))
validate("symbol","symbol must be UPPERCASE ",col("symbol").isNotNull() & (col("symbol") == upper(col("symbol"))))
validate("last_updated","last_updated <= current_timestamp",col("last_updated").isNotNull() & (col("last_updated") <= current_timestamp()))
spark.createDataFrame(validation_results).show(truncate=False)
display(spark.createDataFrame(validation_results))


##adding calculated columns 

In [0]:
#price_vs_ath = current_price / all_time_high
df= df.withColumn("price_vs_ath",when((col("all_time_high") != 0) & col("all_time_high").isNotNull(),col("current_price") / col("all_time_high")))

#volume_to_market_cap = total_volume / market_cap
df =df.withColumn("volume_to_market_cap",when((col("market_cap") != 0) & col("market_cap").isNotNull(), col("total_volume") / col("market_cap")))

##circulating_ratio = circulating_supply / total_supply
df=df.withColumn("circulating_supply_ratio",when((col("total_supply") != 0) & col("total_supply").isNotNull(), col("circulating_supply") / col("total_supply")))

##adding metadata columns


In [0]:
df=df.withColumn("ingestion_time",current_timestamp())
df=df.withColumn("ingestion_date", to_date(current_timestamp()))
df=df.withColumn("source_system",lit('api'))
run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + str(uuid.uuid4())[:6]
df=df.withColumn("pipeline_run_id", lit(run_id))
batch_id= f"batch {run_id}"
df=df.withColumn("batch_id", lit(batch_id))
df=df.withColumn("layer", lit("silver"))

##save to silver

In [0]:
df.write.mode("overwrite").format("delta").partitionBy("ingestion_date").saveAsTable("workspace.silver.crypto")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.silver.crypto LIMIT 10